# NFL Draft Semantic Pillar Scoring

**Pipeline overview:**

| Phase | Step | What it does |
|-------|------|--------------|
| 1 | Normalization | Hyphens/dashes → spaces so `pass-rush` = `pass rush` |
| 1 | Curated stitching | Known NFL compound terms → single underscore tokens |
| 1 | Domain stop words | Keep directional adjectives; remove scouting filler |
| 1 | PMI discovery | Data-driven bigram stitching via NLTK collocations |
| 1 | Lemmatization | WordNet lemmatizer on remaining tokens |
| 2 | TF-IDF | Vectorize cleaned strengths text |
| 2 | Archetype scoring | Cosine similarity to 4 semantic pillar archetypes |
| 2 | 0–100 scaling | Min-max normalize each pillar score for readability |

## 0. Setup

In [ ]:
import re
import warnings
import numpy as np
import pandas as pd
from collections import Counter

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.collocations import BigramCollocationFinder
from nltk.metrics import BigramAssocMeasures

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

for resource in ['punkt', 'stopwords', 'wordnet', 'omw-1.4', 'punkt_tab']:
    nltk.download(resource, quiet=True)

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 80)

# ── Controls ──────────────────────────────────────────────────────────────────
PMI_TOP_N     = 30    # number of auto-discovered bigrams to stitch
PMI_MIN_FREQ  = 5     # minimum corpus frequency for bigram candidates
SCORE_COLS    = ['score_athletic', 'score_technical', 'score_character', 'score_iq']
TOP_N_DISPLAY = 5     # players to show per pillar in summary

## 1. Load Data

Expects a DataFrame `df` with at minimum: `player_name`, `position`, `strengths`.
Replace the sample block below with your own data source.

In [ ]:
# ── Replace this block with your data source ──────────────────────────────────
# Example: df = pd.read_csv('../data/processed/draft_enriched_with_contracts.csv')
#          df = df[['player_name', 'position', 'strengths']].dropna(subset=['strengths'])

# Sample data for standalone demonstration
_sample = [
    ('Micah Parsons',   'LB',   'Elite pass-rush talent with explosive get-off and closing speed. High-motor competitor who plays with relentless effort. Exceptional football IQ and play recognition. Change-of-direction ability is elite.'),
    ('Ja\'Marr Chase',  'WR',   'Contested catch specialist with elite body control and soft hands. Routes are crisp with sharp breaks. Tracks the deep ball exceptionally well. Competitive toughness after the catch.'),
    ('Penei Sewell',    'OT',   'Massive frame with anchor strength and heavy hands. Technically refined with strong footwork and low pad-level. Mauling run-blocker with three-down versatility. High football IQ at the line of scrimmage.'),
    ('Patrick Surtain', 'CB',   'Elite press-coverage cornerback with long arms and explosive recovery speed. High-motor defender who competes on every rep. Ball-hawk instincts with man-coverage dominance. Pre-snap recognition is outstanding.'),
    ('Trevor Lawrence', 'QB',   'Elite arm talent with velocity and touch. Quick-twitch release with exceptional anticipation. Football IQ is elite — reads defenses pre- and post-snap with ease. Leadership and competitive character are unquestioned.'),
    ('Kwity Paye',      'EDGE', 'Pass-rush upside driven by raw athleticism and first-step quickness. Needs to develop counter moves and hand-fighting technique. Motor runs hot. Bend and flexibility are special.'),
    ('Christian Barmore','DT',  'Quick-twitch interior pass rusher. Hand strength and block-shedding ability are NFL-ready. Needs consistency in run defense pad level. Pursuit motor is a strength.'),
    ('Rashawn Slater',  'OT',   'Technically elite left tackle. Footwork and hand placement are advanced. Pass protection is anchor-strength dominant. Football IQ allows him to process stunts and games quickly.'),
]
df = pd.DataFrame(_sample, columns=['player_name', 'position', 'strengths'])
# ─────────────────────────────────────────────────────────────────────────────

df = df.dropna(subset=['strengths']).reset_index(drop=True)
print(f'Players loaded: {len(df)}')
df[['player_name', 'position', 'strengths']].head()

---
## Phase 1 — NFL-Way Preprocessing

### Step 1 — Normalization & Curated Phrase Stitching

In [ ]:
# ── Curated NFL compound terms ─────────────────────────────────────────────────
# Applied AFTER hyphen normalization so 'pass-rush' and 'pass rush' both match.
# Sorted longest-first to prevent partial matches (trigrams before bigrams).
_CURATED_RAW = {
    # Trigrams
    'change of direction':  'change_of_direction',
    'low pad level':        'low_pad_level',
    'run after catch':      'run_after_catch',
    'yards after contact':  'yards_after_contact',
    'yards after catch':    'yards_after_catch',
    'off the line':         'off_the_line',
    'off the ball':         'off_the_ball',
    'point of attack':      'point_of_attack',
    # Bigrams
    'pass rush':            'pass_rush',
    'pass rusher':          'pass_rusher',
    'pass protection':      'pass_protection',
    'pass coverage':        'pass_coverage',
    'pad level':            'pad_level',
    'press coverage':       'press_coverage',
    'man coverage':         'man_coverage',
    'zone coverage':        'zone_coverage',
    'ball skills':          'ball_skills',
    'ball hawk':            'ball_hawk',
    'ball carrier':         'ball_carrier',
    'body control':         'body_control',
    'contact balance':      'contact_balance',
    'closing speed':        'closing_speed',
    'lateral quickness':    'lateral_quickness',
    'quick twitch':         'quick_twitch',
    'high motor':           'high_motor',
    'first step':           'first_step',
    'get off':              'get_off',
    'hand fighting':        'hand_fighting',
    'hand strength':        'hand_strength',
    'block shedding':       'block_shedding',
    'anchor strength':      'anchor_strength',
    'route running':        'route_running',
    'run blocking':         'run_blocking',
    'open field':           'open_field',
    'red zone':             'red_zone',
    'second level':         'second_level',
    'hip flexibility':      'hip_flexibility',
    'soft hands':           'soft_hands',
    'heavy hands':          'heavy_hands',
    'strong hands':         'strong_hands',
    'short area':           'short_area',
    'three down':           'three_down',
    'top end':              'top_end',
    'two gap':              'two_gap',
    'one gap':              'one_gap',
    'snap count':           'snap_count',
}

CURATED_PHRASE_MAP = dict(
    sorted(_CURATED_RAW.items(), key=lambda x: len(x[0]), reverse=True)
)

print(f'Curated phrases defined: {len(CURATED_PHRASE_MAP)}')

### Step 2 — Domain Stop Words

In [ ]:
# Words NLTK would remove that carry real meaning in scouting language
KEEP_WORDS = {
    'high', 'low', 'heavy', 'light', 'deep', 'short', 'long', 'wide',
    'hard', 'soft', 'strong', 'quick', 'good', 'great', 'up', 'down',
    'off', 'out', 'over', 'through', 'above', 'below',
}

# Generic scouting filler — appears in virtually every report regardless of player
CUSTOM_STOPS = {
    'prospect', 'player', 'players', 'show', 'shows', 'need', 'needs',
    'ability', 'also', 'often', 'must', 'well', 'still', 'use', 'get',
    'make', 'look', 'help', 'work', 'time', 'year', 'team', 'game',
    'continue', 'develop', 'development', 'nfl', 'draft', 'college',
    'level', 'type', 'project', 'potential', 'upside', 'ceiling',
}

_base = set(stopwords.words('english'))
NFL_STOPWORDS = (_base - KEEP_WORDS) | CUSTOM_STOPS

print(f'Base NLTK stops : {len(_base)}')
print(f'Un-stopped       : {len(KEEP_WORDS & _base)}  → {sorted(KEEP_WORDS & _base)}')
print(f'Custom added     : {len(CUSTOM_STOPS)}')
print(f'Final stop list  : {len(NFL_STOPWORDS)}')

### Step 3 — Preprocessing Function (applies Steps 1–2 + lemmatization)

In [ ]:
lemmatizer = WordNetLemmatizer()

def nfl_preprocess(text: str, phrase_map: dict = CURATED_PHRASE_MAP,
                   extra_phrases: dict = None) -> str:
    """
    NFL-aware preprocessing pipeline.
    1. Lowercase
    2. Normalize hyphens/dashes → spaces  (MUST be before stitching)
    3. Stitch curated + auto-discovered phrases
    4. Regex clean (keep underscores)
    5. Tokenize → stop word filter → lemmatize → length filter
    """
    if not isinstance(text, str) or not text.strip():
        return ''

    text = text.lower()

    # Step 2a: normalize hyphens and dashes before stitching
    text = re.sub(r'[-\u2013\u2014]', ' ', text)

    # Step 2b: stitch curated phrases
    for phrase, token in phrase_map.items():
        text = text.replace(phrase, token)

    # Step 2c: stitch any PMI-discovered phrases (added after discovery cell)
    if extra_phrases:
        for phrase, token in extra_phrases.items():
            text = text.replace(phrase, token)

    # Step 3: regex — keep letters, underscores, spaces
    text = re.sub(r'[^a-z_\s]', ' ', text)

    # Step 4: tokenize
    tokens = text.split()

    # Step 5: stop word filter (always keep stitched tokens)
    tokens = [t for t in tokens if '_' in t or t not in NFL_STOPWORDS]

    # Step 6: lemmatize non-stitched tokens
    tokens = [t if '_' in t else lemmatizer.lemmatize(t) for t in tokens]

    # Step 7: drop short single tokens
    tokens = [t for t in tokens if len(t) > 1]

    return ' '.join(tokens)


# Initial pass (curated phrases only — PMI phrases added after Step 4)
df['strengths_clean_v1'] = df['strengths'].apply(nfl_preprocess)
print('Preprocessing complete (pre-PMI).')
print('\nSample output:')
for _, row in df.head(3).iterrows():
    print(f"  [{row['position']}] {row['player_name']}")
    print(f"    RAW  : {row['strengths'][:120]}")
    print(f"    CLEAN: {row['strengths_clean_v1'][:120]}")
    print()

### Step 4 — PMI Bigram Discovery

Discover high-PMI bigrams from the corpus that the curated list may have missed.
Review the printed table — the auto-discovered phrases are stitched in the final preprocessing pass.

In [ ]:
# Build token lists from the v1-cleaned text (curated phrases already stitched)
_token_lists = [
    [t for t in text.split() if '_' not in t]   # only un-stitched tokens
    for text in df['strengths_clean_v1']
]

finder = BigramCollocationFinder.from_documents(_token_lists)
finder.apply_freq_filter(PMI_MIN_FREQ)
scored = finder.score_ngrams(BigramAssocMeasures.pmi)

_already = set(CURATED_PHRASE_MAP.keys())

auto_candidates = [
    (w1, w2, round(score, 3), finder.ngram_fd[(w1, w2)])
    for (w1, w2), score in scored
    if  w1 not in NFL_STOPWORDS and w2 not in NFL_STOPWORDS
    and w1.isalpha() and w2.isalpha()
    and len(w1) > 2 and len(w2) > 2
    and f'{w1} {w2}' not in _already
][:PMI_TOP_N]

pmi_df = pd.DataFrame(auto_candidates, columns=['word1', 'word2', 'pmi', 'freq'])
pmi_df['phrase'] = pmi_df['word1'] + ' ' + pmi_df['word2']
pmi_df['token']  = pmi_df['word1'] + '_' + pmi_df['word2']

print(f'Top {PMI_TOP_N} auto-discovered bigrams (PMI ≥ threshold, freq ≥ {PMI_MIN_FREQ}):')
print('These will be stitched in the final preprocessing pass.\n')
print(pmi_df[['phrase', 'token', 'pmi', 'freq']].to_string(index=False))

# Build the auto-phrase map from discovered bigrams
AUTO_PHRASE_MAP = dict(zip(pmi_df['phrase'], pmi_df['token']))
# Sort longest-first (in case of overlapping phrases)
AUTO_PHRASE_MAP = dict(sorted(AUTO_PHRASE_MAP.items(), key=lambda x: len(x[0]), reverse=True))

In [ ]:
# Final preprocessing pass — curated + PMI-discovered phrases
df['strengths_clean'] = df['strengths'].apply(
    lambda t: nfl_preprocess(t, extra_phrases=AUTO_PHRASE_MAP)
)

print('Final preprocessing complete.')
print(f'\nVocabulary size: {len(set(t for text in df["strengths_clean"] for t in text.split())):,} unique tokens')

# Count stitched tokens
_all_tokens = [t for text in df['strengths_clean'] for t in text.split()]
_stitched = Counter(t for t in _all_tokens if '_' in t)
print(f'Stitched phrase tokens found: {len(_stitched)}')
if _stitched:
    print('  Top stitched tokens:')
    for tok, cnt in _stitched.most_common(10):
        print(f'    {tok}: {cnt}')

---
## Phase 2 — Semantic Pillar Scoring

### Step 5 — TF-IDF Vectorization

In [ ]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=1,              # keep rare terms (small corpus); raise to 2-3 for large datasets
    max_df=0.95,           # drop near-universal terms
    sublinear_tf=True,     # dampen high-frequency terms
    token_pattern=r'[a-z_]+',   # allow underscore (stitched tokens)
)

player_matrix = vectorizer.fit_transform(df['strengths_clean'])

print(f'Player matrix : {player_matrix.shape}  (players × features)')
print(f'Vocabulary size: {len(vectorizer.vocabulary_):,}')

# Spot-check: confirm stitched tokens are in vocabulary
_sample_tokens = ['pass_rush', 'pad_level', 'change_of_direction', 'high_motor',
                  'body_control', 'press_coverage', 'route_running', 'anchor_strength']
print('\nStitched tokens in TF-IDF vocabulary:')
for tok in _sample_tokens:
    present = tok in vectorizer.vocabulary_
    print(f'  {tok:30s} {"✓" if present else "–"}')

### Step 6 — Pillar Archetype Definition & Scoring

In [ ]:
# ── 4 Semantic Pillar Archetypes ───────────────────────────────────────────────
# Each string is a dense description of the pillar in scout-like language.
# Using domain terms (including stitched tokens) maximises vocabulary overlap.
ARCHETYPES = {
    'score_athletic': (
        'elite athletic speed explosive burst acceleration lateral quickness '
        'body_control change_of_direction closing_speed explosive_burst '
        'twitch twitchy agile agility flexible flexibility fluid fluidity '
        'vertical jump leap raw physical powerful strength powerful frame '
        'long athletic get_off first_step separation quick_twitch top_end'
    ),
    'score_technical': (
        'technique mechanics footwork hand_fighting hand_strength anchor_strength '
        'pad_level low_pad_level leverage pass_protection run_blocking '
        'route_running block_shedding press_coverage zone_coverage man_coverage '
        'precise precision timing consistent fundamental sound execution '
        'hip_flexibility balance short_area lateral_quickness polish refined '
        'crisp sharp release contested'
    ),
    'score_character': (
        'high_motor motor effort compete relentless hustle competitive '
        'toughness tough grit passion energy aggressive pursuit intensity '
        'leadership character discipline accountable driven focused '
        'diligent committed hard worker never quit battles emotional '
        'physical bully mentality edge'
    ),
    'score_iq': (
        'football iq intelligence smart instinct awareness anticipation '
        'recognition diagnosis read scheme assignment mental processing '
        'pre snap post snap understanding vision cerebral mature prepared '
        'coachable quick processor off_the_ball point_of_attack '
        'chess high football intelligence'
    ),
}

# Transform archetypes using the SAME fitted vectorizer (no refit)
archetype_texts  = [ARCHETYPES[col] for col in SCORE_COLS]
archetype_matrix = vectorizer.transform(archetype_texts)

print('Archetype vectors built.')
for col, vec in zip(SCORE_COLS, archetype_matrix):
    n_nonzero = vec.nnz
    print(f'  {col:25s}  non-zero features: {n_nonzero}')

In [ ]:
# ── Cosine similarity: each player × each archetype ───────────────────────────
sim_matrix = cosine_similarity(player_matrix, archetype_matrix)   # shape: (n_players, 4)

scores_raw = pd.DataFrame(sim_matrix, columns=SCORE_COLS)

# ── Scale 0–100 per pillar (min-max within pillar) ────────────────────────────
scaler = MinMaxScaler(feature_range=(0, 100))
scores_scaled = pd.DataFrame(
    scaler.fit_transform(scores_raw),
    columns=SCORE_COLS
).round(1)

# ── Attach to main DataFrame ──────────────────────────────────────────────────
result = pd.concat(
    [df[['player_name', 'position']].reset_index(drop=True), scores_scaled],
    axis=1
)

print('Pillar scores (0–100 scaled):')
print(result.to_string(index=False))

---
## Output — Results & Pillar Summaries

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ── Final DataFrame ────────────────────────────────────────────────────────────
print('=' * 65)
print('FINAL PILLAR SCORES')
print('=' * 65)
display_cols = ['player_name', 'position'] + SCORE_COLS
print(result[display_cols].sort_values('score_athletic', ascending=False).to_string(index=False))

In [ ]:
# ── Top-5 per pillar ───────────────────────────────────────────────────────────
PILLAR_LABELS = {
    'score_athletic':  'Athletic Profile',
    'score_technical': 'Technical Skills',
    'score_character': 'Competitive Character',
    'score_iq':        'Football IQ',
}

print('TOP 5 PLAYERS PER PILLAR')
print('=' * 65)
for col in SCORE_COLS:
    top = result.nlargest(TOP_N_DISPLAY, col)[['player_name', 'position', col]]
    print(f'\n{PILLAR_LABELS[col].upper()}')
    print(top.to_string(index=False))

In [ ]:
# ── Radar / bar chart: all players across all 4 pillars ───────────────────────
fig, axes = plt.subplots(1, 4, figsize=(18, max(4, len(result) * 0.55 + 1)), sharey=False)

pillar_colors = {
    'score_athletic':  '#4C72B0',
    'score_technical': '#55A868',
    'score_character': '#C44E52',
    'score_iq':        '#8172B2',
}

for ax, col in zip(axes, SCORE_COLS):
    sub = result.sort_values(col, ascending=True)
    bars = ax.barh(
        sub['player_name'] + ' (' + sub['position'] + ')',
        sub[col],
        color=pillar_colors[col],
        alpha=0.85,
        edgecolor='white',
    )
    # Value labels
    for bar, val in zip(bars, sub[col]):
        ax.text(val + 0.5, bar.get_y() + bar.get_height() / 2,
                f'{val:.0f}', va='center', ha='left', fontsize=8)

    ax.set_xlim(0, 115)
    ax.set_xlabel('Score (0–100)')
    ax.set_title(PILLAR_LABELS[col], fontweight='bold', fontsize=10)
    ax.xaxis.set_major_locator(mticker.MultipleLocator(25))
    ax.grid(axis='x', alpha=0.3)
    ax.spines[['top', 'right']].set_visible(False)

fig.suptitle('NFL Draft Semantic Pillar Scores', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()